**<h1>Electric Line Extension - Analysis 2**
#### Descriptive Data
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 09/05/2025 | Start Development: 08/27/2025</i>
* Utility / IOU Data from PG&E, SDG&E, and SCE for 2024
* Goals: 1. Clean excel files for public downloads. See 'Clean 2' below for full details and results


In [263]:
#Import Libraries and Packages
import pandas as pd
import numpy as np
import matplotlib as plt



In [264]:
pge_2024_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/PG&E_Anual_Report_2024.xlsx", header=7)
sdge_2024_MFNC = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SDG&E_ Annual Report_2024.xlsx", sheet_name='Mixed-Fuel New Construction', header=4)
sdge_2024_AENC = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SDG&E_ Annual Report_2024.xlsx", sheet_name='All Electric New Construction', header=4)
sdge_2024_MFU = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SDG&E_ Annual Report_2024.xlsx", sheet_name='Mixed-Fuel Upgrades', header=4)
sdge_2024_AEU = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SDG&E_ Annual Report_2024.xlsx", sheet_name='All Electric Upgrades', header=4)
sce_2024_MFNC = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SCE_Annual_Report_2024.xlsx", sheet_name='Mixed-Fuel New Construction', header=4)
sce_2024_AENC = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SCE_Annual_Report_2024.xlsx", sheet_name='All Electric New Construction', header=4)
sce_2024_MFU = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SCE_Annual_Report_2024.xlsx", sheet_name='Mixed-Fuel Upgrades', header=4)
sce_2024_AEU = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SCE_Annual_Report_2024.xlsx", sheet_name='All Electric Upgrades', header=4)



C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [265]:
#pge_2024_df
#sdge_2024_MFNC.tail(20)
#sce_2024_df

**CLEAN 2**

* For SCE / SDG&E: Create dataframes from each sheet and use the built in function to:
    *   Specifically handle the first 4-15 rows in the raw files which correspond to summaries, totals, and or general info
    *   Iterate through each month recording the month, skipping the repeated headers, and ordering classes
    *   Execute function for each of the four fuel type sheets
* For PG&E: 
    *   Repeat the same steps for the other IOUs however note function is built to handle 'months' rows from raw data as strings instead of as dates
* The shape and columns of all twelve clean dataframes are then verified.
*   Finally, using the melt function, which groups by customer class, and melts on class and month, create final melted dataframes. These are exported as excel spreadsheets and moved to the 'processed' data folder.

In [266]:
def extract_month_and_clean(df, valid_classes, jan_label="Jan 2024", n_jan_rows=4):
    header_name = df.columns[0]
    rows = []
    current_month = None
    row_iter = df.iterrows()
    row_idx = 0

    # Handle January (first n_jan_rows rows)
    for _ in range(n_jan_rows):
        idx, row = next(row_iter)
        first_val = row[header_name]
        if first_val in valid_classes:
            new_row = row.copy()
            new_row['Month'] = jan_label
            rows.append(new_row)
        row_idx += 1

    # Handle the rest of the months
    for _, row in list(row_iter):
        first_val = row[header_name]
        # If this row is a date, update current_month
        try:
            month_val = pd.to_datetime(first_val, errors='raise')
            current_month = month_val.strftime('%b %Y')
            continue  # skip the date row itself
        except Exception:
            pass
        # Skip repeated headers
        if first_val == header_name:
            continue
        # Only keep valid customer class rows
        if first_val in valid_classes and current_month is not None:
            new_row = row.copy()
            new_row['Month'] = current_month
            rows.append(new_row)
    # Create DataFrame and reorder columns
    if rows:
        clean_df = pd.DataFrame(rows)
        cols = ['Month'] + [col for col in clean_df.columns if col != 'Month']
        clean_df = clean_df[cols].reset_index(drop=True)
        return clean_df
    else:
        return pd.DataFrame()  # empty if nothing found


In [267]:
# Execute function on each dataframe to extract month and clean dataframes for both SDG&E and SCE dataframes
valid_classes = ['Residential', 'Commercial', 'Industrial', 'Agriculture', 'Other']
sdge_2024_MFNC_clean = extract_month_and_clean(sdge_2024_MFNC, valid_classes, jan_label="Jan 2024", n_jan_rows=4)
sdge_2024_AENC_clean = extract_month_and_clean(sdge_2024_AENC, valid_classes, jan_label="Jan 2024", n_jan_rows=4)
sdge_2024_MFU_clean = extract_month_and_clean(sdge_2024_MFU, valid_classes, jan_label="Jan 2024", n_jan_rows=4)
sdge_2024_AEU_clean = extract_month_and_clean(sdge_2024_AEU, valid_classes, jan_label="Jan 2024", n_jan_rows=4)

# Skip summary rows for SCE dataframes before cleaning
sce_2024_MFNC_no_summary = sce_2024_MFNC.iloc[7:].reset_index(drop=True)
sce_2024_AENC_no_summary = sce_2024_AENC.iloc[7:].reset_index(drop=True)
sce_2024_MFU_no_summary = sce_2024_MFU.iloc[7:].reset_index(drop=True)
sce_2024_AEU_no_summary = sce_2024_AEU.iloc[7:].reset_index(drop=True)

sce_2024_MFNC_clean = extract_month_and_clean(sce_2024_MFNC_no_summary, valid_classes, jan_label="Jan 2024", n_jan_rows=4)
sce_2024_AENC_clean = extract_month_and_clean(sce_2024_AENC_no_summary, valid_classes, jan_label="Jan 2024", n_jan_rows=4)
sce_2024_MFU_clean = extract_month_and_clean(sce_2024_MFU_no_summary, valid_classes, jan_label="Jan 2024", n_jan_rows=4)
sce_2024_AEU_clean = extract_month_and_clean(sce_2024_AEU_no_summary, valid_classes, jan_label="Jan 2024", n_jan_rows=4)


In [268]:
# Drop Columns with all NaN values from each cleaned dataframe
sce_2024_MFNC_clean = sce_2024_MFNC_clean.dropna(axis=1, how='any')
sce_2024_AENC_clean = sce_2024_AENC_clean.dropna(axis=1, how='any')

In [269]:
# Split each PGE DataFrame into four separate DataFrames based on fuel type: Mixed Fuel New Construction, All Electric Upgrades, Mixed Fuel Upgrades, and All Electric New Construction
pge_2024_df_MFNC = pge_2024_df.iloc[0:169].dropna(how='all').reset_index(drop=True)
pge_2024_df_AEU = pge_2024_df.iloc[175:344].dropna(how='all').reset_index(drop=True)
pge_2024_df_MFU = pge_2024_df.iloc[350:519].dropna(how='all').reset_index(drop=True)
pge_2024_df_AENC = pge_2024_df.iloc[525:694].dropna(how='all').reset_index(drop=True)


In [270]:
def pge_extract_month_and_clean(df, valid_classes, jan_label="Jan 2024", n_jan_rows=9):
    import calendar
    header_name = df.columns[0]
    valid_classes_stripped = [c.strip() for c in valid_classes]
    month_names = [calendar.month_name[i] for i in range(1, 13)]
    rows = []
    idx = 0
    n_rows = len(df)
    # Handle January (first n_jan_rows rows)
    for i in range(n_jan_rows):
        if i >= n_rows:
            break
        row = df.iloc[i]
        first_val = str(row[header_name]).strip()
        if first_val in valid_classes_stripped:
            new_row = row.copy()
            new_row['Month'] = jan_label
            rows.append(new_row)
    idx = n_jan_rows
    # Handle the rest of the months
    while idx < n_rows:
        # Look for a month row
        found_month = False
        while idx < n_rows:
            first_val = str(df.iloc[idx][header_name]).strip()
            # Check if this is a month name (e.g., "February")
            for m in month_names[1:]:  # skip January
                if first_val.lower().startswith(m.lower()):
                    month_label = m[:3] + " 2024"
                    idx += 1
                    found_month = True
                    break
            if found_month:
                break
            idx += 1
        if not found_month:
            break
        # Skip the repeated header row if present
        if idx < n_rows and str(df.iloc[idx][header_name]).strip() == header_name.strip():
            idx += 1
        # Next n_jan_rows are the data rows for this month
        for i in range(n_jan_rows):
            if idx + i >= n_rows:
                break
            row = df.iloc[idx + i]
            first_val = str(row[header_name]).strip()
            if first_val in valid_classes_stripped:
                new_row = row.copy()
                new_row['Month'] = month_label
                rows.append(new_row)
        idx += n_jan_rows
    # Create DataFrame and reorder columns
    if rows:
        clean_df = pd.DataFrame(rows)
        cols = ['Month'] + [col for col in clean_df.columns if col != 'Month']
        clean_df = clean_df[cols].reset_index(drop=True)
        return clean_df
    else:
        return pd.DataFrame()  # empty if nothing found


In [271]:
pge_valid_classes = [
    "Agency (City, County, Caltrans)", "Agricultural", "Commercial", "Industrial",
    "Mixed Use (Commercial / Residential)", "Residential", "Street and Outdoor area Lighting",
    "Telecommunications", "Temporary Services"
]
pge_2024_df_MFNC_clean = pge_extract_month_and_clean(pge_2024_df_MFNC, pge_valid_classes)
pge_2024_df_AENC_clean = pge_extract_month_and_clean(pge_2024_df_AENC, pge_valid_classes)
pge_2024_df_MFU_clean = pge_extract_month_and_clean(pge_2024_df_MFU, pge_valid_classes)
pge_2024_df_AEU_clean = pge_extract_month_and_clean(pge_2024_df_AEU, pge_valid_classes)

pge_2024_df_AENC_clean.head(20)

,Month,Customer Class,Total Discounts (Non‐Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non‐Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non‐Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non‐Refundable,Total Estimated Refundable,Total Electric Line Extension Requests received (applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for applicant install
0,Jan 2024,"Agency (City, County, Caltrans)",287172,0,633714,0,2503,0.0,163070,1208057,84.0,6,16.0
1,Jan 2024,Agricultural,314150,0,1247500,0,11837,0.0,72729,1875800,80.0,56,10.0
2,Jan 2024,Commercial,974229,0,1278131,0,250023,0.0,471820,3226589,181.0,47,39.0
3,Jan 2024,Industrial,141264,0,436684,0,0,0.0,25454,719211,18.0,6,4.0
4,Jan 2024,Mixed Use (Commercial / Residential),56423,0,363650,0,34723,0.0,75721,476496,26.0,7,11.0
5,Jan 2024,Residential,967478,0,2835081,0,0,0.0,2268398,5853932,582.0,216,128.0
6,Jan 2024,Street and Outdoor area Lighting,0,0,8502,0,0,0.0,255200,8502,32.0,3,20.0
7,Jan 2024,Telecommunications,1085982,0,62009,0,0,0.0,394012,2233974,101.0,10,25.0
8,Jan 2024,Temporary Services,58239,0,21501,0,0,0.0,47416,154982,NaN,7,NaN
9,Feb 2024,"Agency (City, County, Caltrans)",194311,0,209992,0,0,NaN,253166,598615,80.0,3,26.0


In [272]:
# Print the column names of each cleaned dataframe to verify the structure and content
print(pge_2024_df_MFNC_clean.columns)
print(sdge_2024_MFNC_clean.columns)
print(sce_2024_MFNC_clean.columns)

Index(['Month', 'Customer Class', 'Total Discounts (Non‐Exempted Projects)',
       'Total Discounts (Exempted projects)',
       'Total Allowances (Non‐Exempted Projects)',
       'Total Allowances (Exempted projects)',
       'Total Refund Payments Provided to Builders (Non‐Exempted Projects)',
       'Total Refund Payments Provided to Builders (Exempted projects)',
       'Total Estimated Non‐Refundable', 'Total Estimated Refundable',
       'Total Electric Line Extension Requests received (applications)',
       'Total Electric Line Extensions Energized',
       'Total Electric Line Extension Applications for applicant install'],
      dtype='object')
Index(['Month', 'Customer Class', 'Total Discounts (Non-Exempted Projects)',
       'Total Discounts (Exempted projects)',
       'Total Allowances (Non-Exempted Projects)',
       'Total Allowances (Exempted projects)',
       'Total Refund Payments Provided to Builders (Non-Exempted Projects)',
       'Total Refund Payments Provided

In [273]:
# Print the shape of each cleaned dataframe to verify the number of rows and columns
print(pge_2024_df_MFNC_clean.shape)
print(pge_2024_df_AEU_clean.shape)
print(pge_2024_df_MFU_clean.shape)
print(pge_2024_df_AENC_clean.shape)
print(sdge_2024_MFNC_clean.shape)
print(sdge_2024_AENC_clean.shape)  
print(sdge_2024_MFU_clean.shape)
print(sdge_2024_AEU_clean.shape)    
print(sce_2024_MFNC_clean.shape)
print(sce_2024_AENC_clean.shape)
print(sce_2024_MFU_clean.shape)
print(sce_2024_AEU_clean.shape) 

(108, 13)
(108, 13)
(108, 13)
(108, 13)
(48, 13)
(48, 13)
(48, 13)
(48, 13)
(48, 13)
(48, 13)
(48, 13)
(48, 13)


In [274]:
def melt_by_customer_class(df, customer_class_col='Customer Class', month_col='Month'):
    """
    Groups by customer class and melts each group, then concatenates the results.
    """
    melted_blocks = []
    for name, group in df.groupby(customer_class_col):
        melted = group.melt(id_vars=[month_col, customer_class_col], var_name='Type', value_name='Count')
        melted_blocks.append(melted)
    return pd.concat(melted_blocks, ignore_index=True)

In [275]:
# Execute the melt function on each cleaned dataframe to create melted dataframes for each IOU
sce_2024_MFNC_melted = melt_by_customer_class(sce_2024_MFNC_clean)
sce_2024_AENC_melted = melt_by_customer_class(sce_2024_AENC_clean)
sce_2024_MFU_melted = melt_by_customer_class(sce_2024_MFU_clean)
sce_2024_AEU_melted = melt_by_customer_class(sce_2024_AEU_clean)
sdge_2024_MFNC_melted = melt_by_customer_class(sdge_2024_MFNC_clean)
sdge_2024_AENC_melted = melt_by_customer_class(sdge_2024_AENC_clean)
sdge_2024_MFU_melted = melt_by_customer_class(sdge_2024_MFU_clean)
sdge_2024_AEU_melted = melt_by_customer_class(sdge_2024_AEU_clean)
pge_2024_df_MFNC_melted = melt_by_customer_class(pge_2024_df_MFNC_clean)
pge_2024_df_AENC_melted = melt_by_customer_class(pge_2024_df_AENC_clean)
pge_2024_df_MFU_melted = melt_by_customer_class(pge_2024_df_MFU_clean)
pge_2024_df_AEU_melted = melt_by_customer_class(pge_2024_df_AEU_clean)

In [276]:
# Export the final melted DataFrames to Excel files, verify, and move to processed folder
# Uncomment to run / reprocess specific files

sce_2024_MFNC_melted.to_excel('SCE Mixed Fuel New Construction 2024.xlsx', sheet_name='Mixed Fuel New Construction 2024', index=False)
sce_2024_AENC_melted.to_excel('SCE All Electric New Construction 2024.xlsx', sheet_name='All Electric New Construction 2024', index=False)
sce_2024_MFU_melted.to_excel('SCE Mixed Fuel Upgrades 2024.xlsx', sheet_name='Mixed Fuel Upgrades 2024', index=False)
sce_2024_AEU_melted.to_excel('SCE All Electric Upgrades 2024.xlsx', sheet_name='All Electric Upgrades 2024', index=False)
sdge_2024_MFNC_melted.to_excel('SDG&E Mixed Fuel New Construction 2024.xlsx', sheet_name='Mixed Fuel New Construction 2024', index=False)
sdge_2024_AENC_melted.to_excel('SDG&E All Electric New Construction 2024.xlsx', sheet_name='All Electric New Construction 2024', index=False)
sdge_2024_MFU_melted.to_excel('SDG&E Mixed Fuel Upgrades 2024.xlsx', sheet_name='Mixed Fuel Upgrades 2024', index=False)
sdge_2024_AEU_melted.to_excel('SDG&E All Electric Upgrades 2024.xlsx', sheet_name='All Electric Upgrades 2024', index=False)
pge_2024_df_MFNC_melted.to_excel('PGE Mixed Fuel New Construction 2024.xlsx', sheet_name='Mixed Fuel New Construction 2024', index=False)
pge_2024_df_AENC_melted.to_excel('PGE All Electric New Construction 2024.xlsx', sheet_name='All Electric New Construction 2024', index=False)
pge_2024_df_MFU_melted.to_excel('PGE Mixed Fuel Upgrades 2024.xlsx', sheet_name='Mixed Fuel Upgrades 2024', index=False)
pge_2024_df_AEU_melted.to_excel('PGE All Electric Upgrades 2024.xlsx', sheet_name='All Electric Upgrades 2024', index=False)


C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
